# Smoke GPU BGE-M3 — RAG Hermes Agent

Relance le smoke d'infrastructure sur un GPU Colab. Objectif : confirmer `rejected_pairs = 0` avec le découpage tokenizer exact.

**Avant de lancer (une seule fois) :**
1. Menu `Exécution` → `Modifier le type d'exécution` → choisir un **GPU** (T4 suffit).
2. Clé 🔑 (panneau de gauche, `Secrets`) → `Ajouter un secret` nommé **`GITHUB_TOKEN`** avec un jeton GitHub en lecture seule de ce dépôt, et active l'accès au notebook.

Ensuite : `Exécution` → `Tout exécuter`.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Cloner le dépôt (privé, via le secret GITHUB_TOKEN)

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
repo = 'devicodeclaude-web/Rag-hermes-agent-'
if not os.path.isdir('/content/Rag-hermes-agent-'):
    os.system(f'git clone https://{token}@github.com/{repo}.git /content/Rag-hermes-agent-')
os.chdir('/content/Rag-hermes-agent-')
print('cwd =', os.getcwd())
print(open('manifests/gpu/bge-m3-smoke-rtx4090.json').readline().strip())

## 3. Installer les dépendances GPU (versions du pyproject)

In [ ]:
!pip install -q FlagEmbedding==1.3.5 transformers==4.57.6 huggingface_hub==0.34.4

## 4. Plan (validation locale, sans téléchargement ni dépense)

In [ ]:
!PYTHONPATH=. python scripts/plan_bge_gpu_job.py

## 5. Lancer le smoke GPU
Télécharge BGE-M3 + reranker aux révisions verrouillées (~4,3 Gio), vérifie les empreintes, découpe au tokenizer exact, encode (dense+sparse) et rerank. Écrit `data/results/bge_m3_gpu_smoke_report.json`.

In [ ]:
!RAG_MODEL_ROOT=/content/models PYTHONPATH=. python scripts/run_bge_gpu_smoke.py

## 6. Résumé du rapport (porte du jalon : `rejected_pairs == 0`)

In [ ]:
import json
r = json.load(open('data/results/bge_m3_gpu_smoke_report.json'))
print('GPU            :', r['hardware']['gpu'], round(r['hardware']['vram_gib'], 1), 'Gio')
print('chunks         :', r['corpus']['chunks'])
print('embedding      :', round(r['embedding']['seconds'], 2), 's  /', round(r['embedding']['chunks_per_second'], 2), 'chunks/s')
print('reranker rejet :', r['reranker']['rejected_pairs'], ' (max tokens par paire =', r['reranker']['maximum_observed_pair_tokens'], ')')
print('recall@10 post :', r['metrics_after_rerank']['recall_at_k'], ' MRR post :', round(r['metrics_after_rerank']['mrr'], 3))
print('fuite          :', r['metrics_after_rerank']['leak_count'])
gate = (r['reranker']['rejected_pairs'] == 0) and (r['metrics_after_rerank']['leak_count'] == 0)
print('\nPORTE JALON (0 rejet, 0 fuite) :', 'OK' if gate else 'ECHEC')

## 7. Récupérer le rapport (télécharge le JSON sur ton appareil)

In [ ]:
from google.colab import files
files.download('data/results/bge_m3_gpu_smoke_report.json')